# EDA 001.05 — Inconsistent Data Entry

**Kaggle Data Cleaning Course — Lesson 5**

Real-world data often has typos, inconsistent casing, and extra whitespace. This notebook covers:

1. Detecting **inconsistencies** (casing, whitespace, typos)
2. Basic cleaning: **strip, lower, replace**
3. **Fuzzy matching** with `fuzzywuzzy` to find similar strings
4. Building a **mapping** to standardize entries
5. Visualizing data before and after cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fuzzywuzzy import fuzz, process

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1 — Load the Dataset

In [ ]:
df = pd.read_csv("data/eda_001_05_researchers.csv")
print(f"Shape: {df.shape}")
df.head(10)

## 2 — Detecting Inconsistencies

How many unique values do we have? (Spoiler: way too many!)

In [ ]:
text_cols = ["country", "city", "university", "field"]

for col in text_cols:
    unique_vals = df[col].nunique()
    print(f"  {col}: {unique_vals} unique values")
    print(f"    Sample: {sorted(df[col].unique())[:8]}")
    print()

In [ ]:
# Visualize: too many categories due to inconsistencies
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, col in zip(axes.flat, text_cols):
    counts = df[col].value_counts()
    counts.plot.barh(ax=ax, color="salmon", edgecolor="white")
    ax.set_title(f"'{col}' — {len(counts)} unique values (BEFORE cleaning)")
    ax.set_xlabel("Count")

plt.suptitle("Inconsistent entries create too many categories!", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3 — Basic Cleaning: Strip Whitespace and Normalize Case

The first line of defense: `str.strip()` + `str.lower()`

In [ ]:
df_clean = df.copy()

for col in text_cols:
    before = df_clean[col].nunique()
    # Strip whitespace and convert to lowercase
    df_clean[col] = df_clean[col].str.strip().str.lower()
    # Also collapse multiple spaces into one
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)
    after = df_clean[col].nunique()
    print(f"  {col}: {before} → {after} unique values")

print("\nRemaining unique values:")
for col in text_cols:
    print(f"  {col}: {sorted(df_clean[col].unique())}")

## 4 — Fuzzy Matching to Find Similar Strings

`fuzzywuzzy` uses **Levenshtein distance** to compute similarity between strings.

Scores range from 0 (completely different) to 100 (identical).

In [ ]:
# Demonstrate fuzzy matching scores
pairs = [
    ("pakistan", "pakstan"),
    ("pakistan", "paksitan"),
    ("india", "inda"),
    ("bangladesh", "bangaldesh"),
    ("sri lanka", "srilanka"),
    ("lahore", "lahor"),
    ("mumbai", "mumbay"),
    ("karachi", "karaci"),
]

print("Fuzzy matching scores:")
print(f"{'String A':<15} {'String B':<15} {'Ratio':>6} {'Partial':>8}")
print("-" * 50)
for a, b in pairs:
    print(f"{a:<15} {b:<15} {fuzz.ratio(a, b):>6} {fuzz.partial_ratio(a, b):>8}")

In [ ]:
# Visualize similarity scores
labels = [f"{a} vs {b}" for a, b in pairs]
ratios = [fuzz.ratio(a, b) for a, b in pairs]
partial_ratios = [fuzz.partial_ratio(a, b) for a, b in pairs]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(labels))
width = 0.35

ax.barh(x - width/2, ratios, width, label="fuzz.ratio", color="steelblue")
ax.barh(x + width/2, partial_ratios, width, label="fuzz.partial_ratio", color="coral")
ax.set_yticks(x)
ax.set_yticklabels(labels)
ax.set_xlabel("Similarity Score (0-100)")
ax.set_title("Fuzzy Matching Scores for Typos")
ax.legend()
ax.axvline(80, color="green", linestyle="--", alpha=0.5, label="Threshold=80")
ax.set_xlim(0, 105)

plt.tight_layout()
plt.show()

## 5 — Building a Correction Map Using Fuzzy Matching

For each unique value, find the **best match** from a list of known correct values.

In [ ]:
def fuzzy_correct(series, correct_values, threshold=80):
    """Map each unique value to the closest correct value if above threshold."""
    unique_vals = series.unique()
    mapping = {}
    
    for val in unique_vals:
        match, score = process.extractOne(val, correct_values)
        if score >= threshold:
            mapping[val] = match
        else:
            mapping[val] = val  # keep as-is if no good match
    
    return mapping

In [ ]:
# Define correct canonical values
correct_countries = ["pakistan", "india", "bangladesh", "sri lanka"]
correct_cities = ["lahore", "karachi", "mumbai", "dhaka", "colombo"]
correct_universities = ["university of punjab", "iit bombay", "university of dhaka", "university of colombo"]
correct_fields = ["computer science", "physics", "mathematics"]

# Build correction maps
country_map = fuzzy_correct(df_clean["country"], correct_countries)
city_map = fuzzy_correct(df_clean["city"], correct_cities)
uni_map = fuzzy_correct(df_clean["university"], correct_universities)
field_map = fuzzy_correct(df_clean["field"], correct_fields)

# Show the mappings
print("Country corrections:")
for k, v in sorted(country_map.items()):
    flag = " ✓" if k == v else " ← CORRECTED"
    print(f"  '{k}' → '{v}'{flag}")

print("\nCity corrections:")
for k, v in sorted(city_map.items()):
    flag = " ✓" if k == v else " ← CORRECTED"
    print(f"  '{k}' → '{v}'{flag}")

In [ ]:
# Apply corrections
df_clean["country"] = df_clean["country"].map(country_map)
df_clean["city"] = df_clean["city"].map(city_map)
df_clean["university"] = df_clean["university"].map(uni_map)
df_clean["field"] = df_clean["field"].map(field_map)

# Title-case for presentation
for col in text_cols:
    df_clean[col] = df_clean[col].str.title()

print("After fuzzy correction:")
for col in text_cols:
    print(f"  {col}: {sorted(df_clean[col].unique())}")

## 6 — Before vs. After: Visual Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flat, text_cols):
    counts = df_clean[col].value_counts()
    counts.plot.barh(ax=ax, color="mediumseagreen", edgecolor="white")
    ax.set_title(f"'{col}' — {len(counts)} unique values (AFTER cleaning)")
    ax.set_xlabel("Count")

plt.suptitle("Clean categories after standardization!", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: unique value counts before vs after
comparison = pd.DataFrame({
    "Before Cleaning": [df[col].nunique() for col in text_cols],
    "After strip+lower": [df[col].str.strip().str.lower().nunique() for col in text_cols],
    "After Fuzzy Match": [df_clean[col].nunique() for col in text_cols],
}, index=text_cols)

fig, ax = plt.subplots(figsize=(10, 5))
comparison.plot.bar(ax=ax, color=["salmon", "gold", "mediumseagreen"], edgecolor="white")
ax.set_title("Unique Values: Before vs. After Cleaning")
ax.set_ylabel("Number of Unique Values")
ax.set_xlabel("Column")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Stage")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation: researchers by country and field
ct = pd.crosstab(df_clean["country"], df_clean["field"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Researchers by Country and Field (after cleaning)")
plt.tight_layout()
plt.show()

In [ ]:
# Publications by field — now meaningful because categories are clean
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_clean, x="field", y="publications", palette="Set2", ax=ax)
ax.set_title("Publication Distribution by Field")
ax.set_xlabel("Field")
ax.set_ylabel("Publications")
plt.tight_layout()
plt.show()

## 7 — Key Takeaways

| Step | Method | Effect |
|------|--------|--------|
| **1. Strip whitespace** | `str.strip()` | Removes leading/trailing spaces |
| **2. Normalize case** | `str.lower()` | Merges "India" / "INDIA" / "india" |
| **3. Collapse spaces** | `str.replace(r'\s+', ' ')` | Fixes "Sri  Lanka" → "Sri Lanka" |
| **4. Fuzzy matching** | `fuzzywuzzy.process.extractOne()` | Corrects typos like "Pakstan" → "Pakistan" |

### Tips
- Always **visualize** category counts before and after cleaning
- Set a **threshold** for fuzzy matching (80+ is a good start) — too low gives false matches
- Maintain a **canonical list** of correct values for each column
- For large datasets, consider `rapidfuzz` (faster drop-in replacement for fuzzywuzzy)
- Log corrections so you can audit what was changed